# Tokenización en LLMs

Cómo los modelos de lenguaje convierten texto en tokens: impacto en coste,
latencia y calidad de las respuestas.

In [ ]:
# pip install tiktoken anthropic
import anthropic, os
import tiktoken  # tokenizador de OpenAI (también útil para estimaciones con otros modelos)

client = anthropic.Anthropic()
MODELO = 'claude-haiku-4-5-20251001'

# Tokenizadores disponibles con tiktoken
enc_gpt4 = tiktoken.encoding_for_model('gpt-4o')
enc_gpt35 = tiktoken.encoding_for_model('gpt-3.5-turbo')
print('Tokenizadores listos')

## 1. ¿Qué es un token?

In [ ]:
def tokenizar_y_mostrar(texto: str, enc=enc_gpt4) -> None:
    tokens = enc.encode(texto)
    decodificados = [enc.decode([t]) for t in tokens]

    print(f'Texto: "{texto}"')
    print(f'Tokens ({len(tokens)}): {tokens}')
    print(f'Decodificados: {decodificados}')
    print()

ejemplos = [
    'Hello',
    'Hola',
    'transformer',
    'transformers',
    'tokenización',
    'Hello, world! How are you today?',
    '2024-01-15',
    '100000',
    'https://www.example.com/api/v2/users',
]

for ej in ejemplos:
    tokenizar_y_mostrar(ej)

## 2. Comparativa de tokenización multilingüe

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# La misma frase en diferentes idiomas
FRASE_BASE = 'La inteligencia artificial está transformando la forma en que trabajamos.'

TRADUCCIONES = {
    'Español':   FRASE_BASE,
    'Inglés':    'Artificial intelligence is transforming the way we work.',
    'Francés':   "L'intelligence artificielle transforme notre façon de travailler.",
    'Alemán':    'Künstliche Intelligenz verändert die Art und Weise, wie wir arbeiten.',
    'Portugués': 'A inteligência artificial está transformando a forma como trabalhamos.',
    'Italiano':  "L'intelligenza artificiale sta trasformando il modo in cui lavoriamo.",
    'Ruso':      'Искусственный интеллект меняет способ нашей работы.',
    'Chino':     '人工智能正在改变我们的工作方式。',
    'Árabe':     'يحول الذكاء الاصطناعي طريقة عملنا.',
    'Japonés':   'AIは私たちの働き方を変えています。',
}

resultados = []
for idioma, texto in TRADUCCIONES.items():
    tokens_gpt4 = len(enc_gpt4.encode(texto))
    chars = len(texto)
    resultados.append({'idioma': idioma, 'texto': texto, 'tokens': tokens_gpt4, 'chars': chars, 'ratio': tokens_gpt4/chars})

resultados.sort(key=lambda x: x['tokens'])

print(f'{"Idioma":<12} {"Caracteres":>12} {"Tokens GPT-4":>14} {"Tokens/Char":>12}')
print('-' * 55)
for r in resultados:
    print(f'{r["idioma"]:<12} {r["chars"]:>12} {r["tokens"]:>14} {r["ratio"]:>12.3f}')

print('\n⚠️  Los idiomas con menos tokens en el vocabulario (árabe, chino, japonés)')
print('    consumen más tokens por carácter → más caro y más lento')

## 3. Impacto del formato en el número de tokens

In [ ]:
# El mismo dato representado de formas diferentes puede costar muy distinto en tokens
import json

datos_ejemplo = {
    'usuario_id': 12345,
    'nombre': 'Ana García',
    'email': 'ana@empresa.com',
    'plan': 'pro',
    'tokens_usados': 450000,
    'fecha_registro': '2025-03-15',
}

representaciones = {
    'JSON completo': json.dumps(datos_ejemplo, ensure_ascii=False),
    'JSON compacto': json.dumps(datos_ejemplo, separators=(',', ':'), ensure_ascii=False),
    'Solo valores':  ' | '.join(str(v) for v in datos_ejemplo.values()),
    'Markdown':      '| ' + ' | '.join(datos_ejemplo.keys()) + ' |\n| ' + ' | '.join(str(v) for v in datos_ejemplo.values()) + ' |',
    'Texto natural': f'El usuario {datos_ejemplo["nombre"]} ({datos_ejemplo["email"]}) tiene el plan {datos_ejemplo["plan"]} y ha usado {datos_ejemplo["tokens_usados"]:,} tokens desde {datos_ejemplo["fecha_registro"]}.',
}

print(f'{"Formato":<20} {"Tokens":>8} {"Chars":>8} {"Eficiencia":>12}')
print('-' * 55)
for nombre, texto in representaciones.items():
    tokens = len(enc_gpt4.encode(texto))
    chars = len(texto)
    eficiencia = chars / tokens  # chars por token (más alto = más eficiente)
    print(f'{nombre:<20} {tokens:>8} {chars:>8} {eficiencia:>11.2f}c/t')

print('\n💡 JSON compacto es ~20% más barato que JSON con espacios')
print('   Para datos estructurados en prompts, usa JSON compacto')

## 4. Contar tokens con la API de Anthropic

In [ ]:
def auditar_prompt(system: str, mensajes: list[dict]) -> dict:
    """Audita el coste en tokens de un prompt antes de enviarlo."""
    resp = client.messages.count_tokens(
        model=MODELO,
        system=system,
        messages=mensajes,
    )
    tokens_entrada = resp.input_tokens

    # Estimaciones de coste
    coste_haiku = tokens_entrada * 0.80 / 1_000_000
    coste_sonnet = tokens_entrada * 3.00 / 1_000_000

    return {
        'tokens_entrada': tokens_entrada,
        'coste_haiku_usd': coste_haiku,
        'coste_sonnet_usd': coste_sonnet,
        'pct_ventana_200k': tokens_entrada / 200_000 * 100,
    }

# Auditar diferentes prompts
system_corto = 'Eres un asistente útil.'
system_largo = '''Eres un experto en análisis de contratos legales con 20 años de experiencia.
Tu especialidad es el derecho mercantil español y europeo. Siempre identificas:
1. Cláusulas de riesgo y responsabilidad
2. Condiciones de resolución anticipada
3. Obligaciones de confidencialidad
4. Condiciones económicas y penalizaciones
Responde siempre en español y cita artículos específicos del contrato.'''

mensajes_prueba = [{
    'role': 'user',
    'content': 'Analiza este contrato: ' + 'Lorem ipsum... ' * 100  # simular texto largo
}]

for nombre, system in [('System corto', system_corto), ('System largo', system_largo)]:
    auditoria = auditar_prompt(system, mensajes_prueba)
    print(f'{nombre}:')
    print(f'  Tokens: {auditoria["tokens_entrada"]:,}')
    print(f'  Coste Haiku: ${auditoria["coste_haiku_usd"]:.6f}')
    print(f'  Coste Sonnet: ${auditoria["coste_sonnet_usd"]:.6f}')
    print(f'  % ventana usada: {auditoria["pct_ventana_200k"]:.2f}%\n')

## 5. Optimización de prompts para reducir tokens

In [ ]:
PROMPT_VERBOSO = """
Por favor, me gustaría que analizaras el siguiente texto que te voy a proporcionar a continuación.
Necesito que identifiques el sentimiento general del texto, que puede ser positivo, negativo o neutro.
También me gustaría que me dijeras cuál es el tema principal que se trata en el texto.
Por favor, devuelve tu respuesta en formato JSON con los campos 'sentimiento' y 'tema'.
El texto que debes analizar es el siguiente:
"""

PROMPT_COMPACTO = """
Analiza el texto. JSON: {"sentimiento": "positivo|negativo|neutro", "tema": "..."}.
Texto:
"""

TEXTO_TEST = 'El nuevo modelo de IA ha superado todas las expectativas en las pruebas de rendimiento.'

mensajes_verboso  = [{'role': 'user', 'content': PROMPT_VERBOSO + TEXTO_TEST}]
mensajes_compacto = [{'role': 'user', 'content': PROMPT_COMPACTO + TEXTO_TEST}]

tokens_verboso  = client.messages.count_tokens(model=MODELO, messages=mensajes_verboso).input_tokens
tokens_compacto = client.messages.count_tokens(model=MODELO, messages=mensajes_compacto).input_tokens

print(f'Prompt verboso:  {tokens_verboso:>4} tokens')
print(f'Prompt compacto: {tokens_compacto:>4} tokens')
print(f'Ahorro: {tokens_verboso - tokens_compacto} tokens ({(1 - tokens_compacto/tokens_verboso)*100:.1f}%)')

# Verificar que ambos producen el mismo resultado
resp = client.messages.create(
    model=MODELO, max_tokens=50, temperature=0,
    messages=mensajes_compacto
)
print(f'\nResultado compacto: {resp.content[0].text}')